# 1. Base Setup

## 1.1 Install packages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install kagglehub
!pip install statsmodels

## 1.2 Load all needed imports

In [ ]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [ ]:
import cf_copilot
print(cf_copilot.__file__)

In [ ]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [ ]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.8)

train_df = big_df[big_df["reference_date"] <= cutoff_date]
test_df = big_df[big_df["reference_date"] > cutoff_date]

X_train, y_train = preprocess(train_df)
X_test, y_test = preprocess(test_df)

pipeline = initialize_model()
pipeline = pipeline.fit(X_train, y_train)

# 3. Feature selection

## 3.1 show feature importance

In [ ]:
def show_feature_importance(pipeline):
    all_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

    # Filter by which ones survived VarianceThreshold
    if "variance" in pipeline.named_steps:
        kept_mask = pipeline.named_steps["variance"].get_support()
        feature_names = all_names[kept_mask]
        feature_names = [name.split("__", 1)[1] for name in feature_names]
    else:
        feature_names = all_names


    importances = pd.DataFrame({
        "feature": feature_names,
        "importance": pipeline.named_steps["classifier"].feature_importances_
    }).sort_values("importance", ascending=False)
    return importances

In [ ]:
df = show_feature_importance(pipeline)
df

In [ ]:
from cf_copilot.ml_logic.evaluation import evaluate_training_run

def evaluate(pipe):
    evaluation_results = evaluate_training_run(pipeline=pipe, X_test=X_test, y_test=y_test,
                                                test_df=test_df, big_df=big_df, log_backtests_to_mlflow=False
    )
    return {
        'metrics': evaluation_results["metrics"],
        # 'figures': evaluation_results["figures"],
        # 'artifacts': evaluation_results["artifacts"],
        # 'forecast_summary': evaluation_results["forecast_summary"],
        # 'backtest_summary': evaluation_results["backtest_summary"],
        'forecast_backtest_summary': evaluation_results["forecast_backtest_summary"]
    }

evaluation = evaluate(pipeline)

In [ ]:
evaluation

## 3.2  Add variance

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Insert VarianceThreshold in existing pipeline, this drops near-constant features
new_pipeline = Pipeline([
    ("preprocessor", pipeline.named_steps["preprocessor"]),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", pipeline.named_steps["classifier"]),
])
new_pipeline.fit(X_train, y_train)
show_feature_importance(new_pipeline)


In [ ]:
evaluate(new_pipeline)

## 3.3 Permutation importance

### Permutation Importance — What does it mean?

Permutation importance works by shuffling one feature at a time and checking how much the model's performance drops. If the score drops a lot, that feature is important. If nothing changes, the feature isn't doing anything useful.

How to read the values:
- **importance_mean > 0** (e.g. 0.05): shuffling this feature made the model worse by 0.05. Good — it means the feature matters. Higher = more important.
- **importance_mean ≈ 0** (e.g. 0.001): the feature barely contributes anything. Could probably drop it.
- **importance_mean < 0** (e.g. -0.01): the model actually got *better* when we shuffled this feature. It's just adding noise — should definitely drop it.

importance_std shows how stable the result is across repeats. If importance_std is bigger than importance_mean, the result isn't very reliable and we can't really trust that feature's score.

One thing to watch out for: if two features are highly correlated, they'll both show lower importance than expected because shuffling one still leaves the other intact. So it's worth checking multicollinearity before drawing conclusions from this.

We're using X_test here instead of X_train to avoid overfit features looking more important than they actually are.

In [ ]:
from sklearn.inspection import permutation_importance
X_pre = pipeline.named_steps["preprocessor"].fit_transform(X_train, y_train)

perm = permutation_importance(new_pipeline, X_train, y_train, n_repeats=10, scoring='neg_log_loss')

In [ ]:
feature_names = new_pipeline.named_steps["preprocessor"].get_feature_names_out()

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_df

## 3.4 Automatic feature selection with CrossValidate

### RFECV — Recursive Feature Elimination with Cross Validation

RFECV works by repeatedly removing the least important feature, refitting the model each time, and checking the cross-validated score. It keeps doing this until it finds the smallest set of features that still gives the best performance.

How to read the results:
- **rfecv.n_features_**: the optimal number of features it found.
- **rfecv.support_**: a boolean mask showing which features were kept (True) and which were dropped (False).
- **rfecv.ranking_**: ranking per feature — 1 means it was selected, higher numbers mean it was eliminated earlier.
- **rfecv.cv_results_["mean_test_score"]**: the CV score at each step, useful to plot and see where performance peaks.



If the curve flattens out early, it means a lot of features aren't really helping and we can safely use a smaller set. If it keeps climbing until the end, we probably need all of them.

In [ ]:
# This takes quite some time with our current number of features, so be aware

from sklearn.feature_selection import RFECV
X_pre = pipeline.named_steps["preprocessor"].fit_transform(X_train, y_train)

rfecv = RFECV(
    estimator=pipeline.named_steps["classifier"],
    step=1,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)fff
rfecv.fit(X_pre, y_train)
X_selected = rfecv.transform(X_train)

rfecv_df = pd.DataFrame({
    "feature": feature_names,
    "selected": rfecv.support_,
    "ranking": rfecv.ranking_,
}).sort_values("ranking")

rfecv_df

In [ ]:
all_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print(all_names)

In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
selected_names = feature_names[rfecv.support_]
selected_names = [name.split("__", 1)[1] for name in selected_names]

X_selected_df = pd.DataFrame(X_selected, columns=selected_names)
X_selected_df.head()

# 4. Multicoliniarity

How to interpret VIF:

- VIF < 5 → fine
- VIF 5–10 → moderate, keep an eye on it
- VIF > 10 → strong multicollinearity, consider dropping one of the correlated pair

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_pre = pipeline.named_steps["preprocessor"].transform(X_train)
feature_names = [n.split("__", 1)[1] for n in pipeline.named_steps["preprocessor"].get_feature_names_out()]

vif_df = pd.DataFrame({
    "feature": feature_names,
    "VIF": [variance_inflation_factor(X_pre, i) for i in range(X_pre.shape[1])]
}).sort_values("VIF", ascending=False)

vif_df